# DeepPurpose CNN

SMILES CNN model for LD50 regression.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
from DeepPurpose import CompoundPred, utils

PROJECT_ROOT = Path('../..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import TARGET_COLUMN, SMILES_COLUMN, artifact_paths, load_smiles_splits, plot_pred_vs_real, regression_metrics, save_predictions


In [ ]:
RUN_ID = 'deeppurpose_cnn'
DRUG_ENCODING = 'CNN'
splits = load_smiles_splits()
paths = artifact_paths(Path.cwd(), RUN_ID, '', include_metadata=True)


In [ ]:
train_data = utils.data_process(X_drug=splits['train'][SMILES_COLUMN].values, y=splits['train'][TARGET_COLUMN].values, drug_encoding=DRUG_ENCODING, split_method='no_split')
valid_data = utils.data_process(X_drug=splits['valid'][SMILES_COLUMN].values, y=splits['valid'][TARGET_COLUMN].values, drug_encoding=DRUG_ENCODING, split_method='no_split')
test_data = utils.data_process(X_drug=splits['test'][SMILES_COLUMN].values, y=splits['test'][TARGET_COLUMN].values, drug_encoding=DRUG_ENCODING, split_method='no_split')

config = utils.generate_config(drug_encoding=DRUG_ENCODING, train_epoch=20, LR=0.001, batch_size=128)
model = CompoundPred.model_initialize(**config)
model.train(train_data, valid_data, test_data)
predictions = np.asarray(model.predict(test_data)).ravel()


In [ ]:
y_test = splits['test'][TARGET_COLUMN].to_numpy()
metrics = regression_metrics(y_test, predictions)
model.save_model(str(paths['model']))
save_predictions(paths['predictions'], y_test, predictions)
plot_pred_vs_real(paths['pred_vs_real'], y_test, predictions, 'DeepPurpose CNN: Predicted vs Real')
with paths['metadata'].open('w', encoding='utf-8') as fp:
    json.dump({'run_id': RUN_ID, 'drug_encoding': DRUG_ENCODING, 'metrics': metrics}, fp, indent=2)
metrics
